In [1]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt

# ================================
# 2. UPLOAD DATASET
# ================================
from google.colab import files
uploaded = files.upload()

data = pd.read_csv(next(iter(uploaded)))

# ================================
# 3. AGE GROUP SEGMENTATION
# ================================
def age_group(age):
    if age <= 18:
        return "Paediatrics"
    elif age <= 45:
        return "Early Adulthood"
    elif age <= 64:
        return "Middle Age"
    else:
        return "Geriatrics"

data['Age_Group'] = data['Age'].apply(age_group)

# ================================
# 4. DATA PREPROCESSING
# ================================
data = data.dropna()
data = data.drop_duplicates()

# Encode categorical variables
le = LabelEncoder()
categorical_cols = data.select_dtypes(include=['object']).columns

for col in categorical_cols:
    data[col] = le.fit_transform(data[col])

# Feature scaling
scaler = StandardScaler()
numerical_cols = data.select_dtypes(include=['int64','float64']).columns
numerical_cols = numerical_cols.drop('Type of Diabetes')

data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

# ================================
# 5. HANDLE IMBALANCED DATA
# ================================
majority = data[data['Type of Diabetes'] == 0]
minority = data[data['Type of Diabetes'] != 0]

minority_upsampled = resample(minority,
                             replace=True,
                             n_samples=len(majority),
                             random_state=42)

data_balanced = pd.concat([majority, minority_upsampled])

# ================================
# 6. FEATURES & TARGET
# ================================
X = data_balanced.drop('Type of Diabetes', axis=1)
y = data_balanced['Type of Diabetes']

# ================================
# 7. MODELS
# ================================
models = {
    "RF": RandomForestClassifier(),
    "DT": DecisionTreeClassifier(),
    "ET": ExtraTreesClassifier(),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "Bagging": BaggingClassifier(),
    "LR": LogisticRegression(max_iter=1000),
    "NB": GaussianNB(),
    "GB": GradientBoostingClassifier()
}

# ================================
# 8. DATA SPLITS
# ================================
splits = {
    "80:20": 0.2,
    "70:30": 0.3,
    "60:40": 0.4
}

# ================================
# 9. TRAINING + EVALUATION
# ================================
results = []

for split_name, test_size in splits.items():

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')

        results.append([split_name, name, acc, prec, rec])

# ================================
# 10. RESULTS TABLE
# ================================
results_df = pd.DataFrame(results, columns=[
    "Split", "Model", "Accuracy", "Precision", "Recall"
])

print(results_df)

# ================================
# 11. HEATMAPS
# ================================
pivot_acc = results_df.pivot(index="Model", columns="Split", values="Accuracy")
pivot_prec = results_df.pivot(index="Model", columns="Split", values="Precision")
pivot_rec = results_df.pivot(index="Model", columns="Split", values="Recall")

plt.figure(figsize=(18,5))

plt.subplot(1,3,1)
sns.heatmap(pivot_acc, annot=True, fmt=".3f", cmap="coolwarm")
plt.title("Accuracy")

plt.subplot(1,3,2)
sns.heatmap(pivot_prec, annot=True, fmt=".3f", cmap="coolwarm")
plt.title("Precision")

plt.subplot(1,3,3)
sns.heatmap(pivot_rec, annot=True, fmt=".3f", cmap="coolwarm")
plt.title("Recall")

plt.tight_layout()
plt.show()

# ================================
# 12. CONFUSION MATRIX (BEST MODEL)
# ================================
best_model = RandomForestClassifier()
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.title("Confusion Matrix ()")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

# ================================
# 13. FEATURE IMPORTANCE
# ================================
importances = best_model.feature_importances_
features = X.columns

feat_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Top 10 features
top_features = feat_df.head(10)

plt.figure(figsize=(8,5))
sns.barplot(x='Importance', y='Feature', data=top_features)

plt.title("Top 10 Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Features")

plt.show()


KeyboardInterrupt: 